# Forget-MI LoKU — Machine Unlearning Pipeline (UNIFIED)

> ✅ **Notebook hợp nhất** — 3 cell training riêng cho **3% / 6% / 10%**, chạy theo nhu cầu.
> Config dùng chung: `exp11_final_ihl075` (honest, IHL=0.75, image-FILA, distill từ F_og).
> Train dừng theo `val_CE` (KHÔNG đụng `F_re` khi train).

## Cấu trúc notebook

| Cell | Mục đích | Khi chạy |
|---|---|---|
| 1 | Mount Drive + pull code | Mọi session |
| 2 | Extract data & models | Lần đầu |
| 3 | Preprocess + symlink output | Lần đầu |
| 3.5 | Verify config + integrity check | Mỗi session (5s) |
| 3.6 | **Định nghĩa helpers** (`run_multiseed`, `aggregate_summary`) | **BẮT BUỘC** trước 4a/4b/4c |
| 4a | Train multi-seed FORGET 3% | Theo nhu cầu |
| 4b | Train multi-seed FORGET 6% | Theo nhu cầu |
| 4c | Train multi-seed FORGET 10% | Theo nhu cầu |
| 5 | Bảng tổng hợp 3 forget% (Bảng 1 luận văn) | Sau khi train xong các cell 4* |
| 6 | Auto-commit & push lên GitHub | Sau khi có kết quả mới |

## ⚠️ Lưu ý gold retrained model

- **3%**: ✅ có `model_retrained_3per/` (paper release) → 1−CosSim hợp lệ
- **6% / 10%**: ❌ chưa retrain → 1−CosSim **KHÔNG hợp lệ**, các metric khác (MIA / Df / Dt / CE / time / GPU) vẫn đúng

> 📌 Notebook cũ `run_6per.ipynb` và `run_10per.ipynb` **đã được merge vào đây** — có thể xóa.

In [ ]:
# ====================================
# CELL 1: Kết nối Drive & Pull Code (giữ data, không clone lại)
# ====================================
from google.colab import drive
import os

# 1. Mount Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive đã được kết nối!")

# 2. Pull code mới (KHÔNG xóa thư mục → data đã extract được giữ nguyên)
%cd /content
REPO = "Forget-MI-LoKU"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"

if not os.path.exists(REPO):
    print(f"🔽 Clone lần đầu: {REPO}")
    !git clone {REPO_URL}
else:
    print(f"🔄 Pull code mới (giữ data đã extract)")
    %cd {REPO}
    !git fetch origin
    !git reset --hard origin/master 2>&1 | tail -3
    %cd /content

%cd {REPO}
!git log --oneline -1

# 3. Cài đặt thư viện (chỉ chạy lần đầu hoặc khi cần update)
import importlib.util
need_install = importlib.util.find_spec("peft") is None or importlib.util.find_spec("pydicom") is None
if need_install:
    print("📦 Cài đặt thư viện...")
    !pip install -q pydicom scikit-image wandb pyyaml pandas
    !pip install -q "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"
else:
    print("✅ Thư viện đã cài, bỏ qua.")

# 4. ⚠️ KIỂM TRA GPU — BẮT BUỘC. Không có GPU → mỗi exp chậm ~25-30× (train treo ở Epoch 0).
import torch
if torch.cuda.is_available():
    print(f"\n🟢 GPU OK: {torch.cuda.get_device_name(0)}")
else:
    print("\n" + "!"*60)
    print("🔴 KHÔNG CÓ GPU — Fisher sẽ ~12 phút, train gần như TREO ở Epoch 0!")
    print("   → Colab: Runtime ▸ Change runtime type ▸ Hardware accelerator = GPU (T4) ▸ Reconnect")
    print("   → Free-tier hết hạn mức GPU cũng bị đẩy về CPU (đợi reset / đổi acc / Colab Pro).")
    print("   → ĐỪNG chạy Cell 4* khi còn dòng đỏ này.")
    print("!"*60)

print("\n✅ Môi trường và mã nguồn đã sẵn sàng!")
print("ℹ️  Lần đầu: chạy Cell 2 → Cell 3 (extract data).")
print("ℹ️  Lần sau: bỏ qua Cell 2 + Cell 3, đi thẳng Cell 3.5 → Cell 3.6 → Cell 4* → Cell 5/6.")

In [ ]:
# ====================================
# CELL 2: Giải nén Data & Models (CHỈ CHẠY LẦN ĐẦU)
# ====================================
!python setup_data.py

In [ ]:
# ====================================
# CELL 3: Tiền xử lý & Thiết lập Output (CHỈ CHẠY LẦN ĐẦU)
# ====================================
import os
import shutil

# 1. Tạo all_data.tsv từ các file báo cáo (idempotent)
!python make_tsv.py

# 2. Cache features — KHÔNG xóa nữa (xóa = phải regenerate 5+ phút mỗi lần)
#    Uncomment 2 dòng dưới nếu THỰC SỰ muốn force regenerate cache
# !rm -f ./data/metadata/cachedfeatures_train_seqlen-*
# !rm -f ./data/metadata/cachednoisyfeatures_train_seqlen-*

# 3. Kết nối thư mục Output với Drive để lưu bền vững
DRIVE_RESULTS = "/content/drive/MyDrive/Forget-MI-Project/unlearning_output"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if os.path.exists("unlearning_output"):
    if os.path.islink("unlearning_output"):
        os.unlink("unlearning_output")
    else:
        shutil.rmtree("unlearning_output")

!ln -s "{DRIVE_RESULTS}" ./unlearning_output

# 4. Verify cache files có sẵn không
cache_dir = "./data/metadata"
has_cache = False
if os.path.exists(cache_dir):
    files = os.listdir(cache_dir)
    has_cache = any(f.startswith(("cachedfeatures_train_seqlen", "cachednoisyfeatures_train_seqlen"))
                    for f in files)

print(f"\n✅ Output sẽ được lưu tại: {DRIVE_RESULTS}")
print(f"{'✅' if has_cache else '⚠️ '} Cache features {'đã có' if has_cache else 'CHƯA có'} trong {cache_dir}")
if not has_cache:
    print("   → Cell 4* lần đầu sẽ chậm (~5 phút regenerate features). Lần sau sẽ load cache nhanh.")

In [ ]:
# ====================================
# CELL 3.5: Verify config + INTEGRITY CHECK (chạy mỗi session)
# ====================================
print("📋 Config hiện tại (các tham số hay đổi giữa các exp):\n")
!grep -E "^\s*(forget_margin|alpha|beta|theta|gamma|lora_r|unlearn_epochs|learning_rate|kappa_cls_retain|kappa_cls_forget|distill_teacher|distill_retain_weight|distill_forget_weight|loku_subtract_scale|ihl_forget_weight|lora_image_last_k_blocks|loku_image_subtract_scale|early_stop_metric|eta_re_anchor):" -A 1 config.yaml | grep -v "^--"

print("\n🧼 INTEGRITY CHECK — KHÔNG được dùng F_re khi train (bản honest exp11):")
!grep -E "^\s*(distill_teacher|early_stop_metric|distill_forget_weight|eta_re_anchor):" -A 1 config.yaml | grep "value:" | xargs -I{} echo "   → {}"
print("   ✅ ĐÚNG khi: distill_teacher=og, early_stop_metric=val, distill_forget_weight=0, eta_re_anchor=0")
print("   ⚠️  Nếu distill_teacher=re HOẶC early_stop_metric=cossim → đang DÙNG F_re (chỉ hợp lệ cho exp10c).")

print("\n🧪 Code có cơ chế honest (teacher=og + early-stop val):")
!grep -c "use_og_teacher\|early_stop_metric\|val_subset\|resolve_image_targets\|_fila_decompose\|TRUE-SUBTRACTION" training/forgetmi_loku.py | xargs -I{} echo "   → khớp pattern: {} (nên >= 6)"

print("\n🔍 Git commit đang chạy:")
!git log --oneline -1

print("\n👉 Nếu config CŨ → local `git push` rồi rerun Cell 1.")

In [ ]:
# ====================================
# CELL 3.6: Helpers cho training + evaluation (BẮT BUỘC trước Cell 4*)
# ====================================
# Định nghĩa hai helper dùng chung:
#   • run_multiseed(forget_pct, seeds, ihl)  — train + auto-aggregate
#   • aggregate_summary(forget_pct, seeds, ...) — chỉ aggregate (không train)
# CHỈ ĐỊNH NGHĨA — không chạy training.
#
# Naming convention (cleaned 2026-06-16):
#   • Individual exp MD: tracker tự gán `exp_NNN_<slug>.md` (NNN = auto ID).
#     KHÔNG đặt `exp_name` bắt đầu bằng "exp..." → tracker._slug() sẽ strip
#     "exp\\d*[a-z]*_" để tránh double-prefix bug (`exp_NNN_expMM_...`).
#   • Summary MD: `summary_<slug>_multiseed.md` (không có ID, vì gom nhiều exp).

import os, numpy as np, pandas as pd
from datetime import datetime

CSV_PATH = "unlearning_output/results_summary.csv"

# Reference numbers từ paper Forget-MI Table 1 (Hardan et al. MICCAI 2024)
PAPER_REF = {
    3:  {"MIA_paper": 0.571, "Df_AUC": 0.735, "Df_F1": 0.393, "Dt_AUC": 0.625, "Dt_F1": 0.250, "Time_h": 5.0},
    6:  {"MIA_paper": 0.615, "Df_AUC": 0.654, "Df_F1": 0.328, "Dt_AUC": 0.599, "Dt_F1": 0.270, "Time_h": 5.0},
    10: {"MIA_paper": 0.810, "Df_AUC": 0.656, "Df_F1": 0.313, "Dt_AUC": 0.565, "Dt_F1": 0.252, "Time_h": 5.0},
}

# Gold retrained models — chỉ có 3% (paper release). 6%/10% phải tự retrain.
GOLD_RETRAINED = {
    3:  "./model_retrained_3per/",
    6:  None,   # chưa retrain → 1-CosSim sẽ KHÔNG hợp lệ
    10: None,   # chưa retrain → 1-CosSim sẽ KHÔNG hợp lệ
}

# Metric definitions: (csv_key, display, paper_key, direction)
# direction: '↓' = thấp tốt, '↑' = cao tốt, '·' = neutral/diagnostic
METRIC_DEFS = [
    ('MIA',                 'MIA_persample',   None,         '↓'),
    ('MIA_paper',           'MIA_paper',       'MIA_paper',  '↓'),
    ('forget_ce',           'forget_ce',       None,         '·'),
    ('test_ce',             'test_ce',         None,         '·'),
    ('Df_AUC',              'Forget AUC',      'Df_AUC',     '↓'),
    ('Df_F1',               'Forget Mac-F1',   'Df_F1',      '↓'),
    ('Dt_AUC',              'Test AUC',        'Dt_AUC',     '↑'),
    ('Dt_F1',               'Test Mac-F1',     'Dt_F1',      '↑'),
    ('dist_vs_re',          '1 − CosSim',      None,         '↓'),
    ('unlearn_time_hours',  'Time (h)',        'Time_h',     '↓'),
    ('gpu_peak_GB',         'GPU peak (GB)',   None,         '·'),
    ('trainable_ratio',     'Trainable ratio', None,         '↓'),
]


def _check_gold(forget_pct):
    """Return (path, valid) — valid=False nếu path không tồn tại trên disk."""
    p = GOLD_RETRAINED.get(forget_pct)
    if p is None:
        return None, False
    return p, os.path.isdir(p)


def _seed_done(forget_pct, seed):
    """Check xem (forget_pct, seed) đã có trong CSV chưa."""
    if not os.path.exists(CSV_PATH):
        return False
    try:
        df = pd.read_csv(CSV_PATH)
    except Exception:
        return False
    if 'forget_pct' not in df.columns or 'seed' not in df.columns:
        return False
    mask = df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per") & (df['seed'] == seed)
    return bool(mask.any())


def run_multiseed(forget_pct, seeds=(42, 123, 7), ihl=0.75, force_redo=False):
    """Train multi-seed tại forget_pct cụ thể.

    Args:
        forget_pct (int): 3, 6, or 10
        seeds (tuple): random seeds (default 3 seeds chính thức)
        ihl (float): IHL weight (default 0.75 = sweet-spot từ exp11d)
        force_redo (bool): chạy lại dù seed đã có trong CSV
    """
    assert forget_pct in (3, 6, 10), f"forget_pct phải 3/6/10, nhận {forget_pct}"

    forget_csv = f"./data_splits/forget_set_{forget_pct}per.csv"
    retrained_path, has_gold = _check_gold(forget_pct)
    paper = PAPER_REF[forget_pct]
    # NOTE: exp_name KHÔNG có prefix 'exp' — tracker tự gán `exp_NNN_` khi tạo file MD.
    # Slug logic ở scripts/exp_tracker.py:_slug() sẽ strip 'exp\d*[a-z]*_' đầu chuỗi.
    exp_name = f"final_ihl{int(ihl*100):03d}_{forget_pct}per"

    # ---- Banner ----
    print(f"\n{'#'*72}")
    print(f"# 🎯 FORGET {forget_pct}% — multi-seed {list(seeds)}, IHL={ihl}")
    print(f"# Forget CSV    : {forget_csv}")
    print(f"# Run name      : {exp_name}  (→ MD: exp_NNN_{exp_name}_seed{{S}}.md)")
    if has_gold:
        print(f"# Gold retrained: ✅ {retrained_path} → 1−CosSim HỢP LỆ")
    else:
        gold_str = GOLD_RETRAINED.get(forget_pct) or 'N/A'
        print(f"# Gold retrained: ❌ {gold_str} (chưa có cho {forget_pct}%)")
        print(f"#   → 1−CosSim sẽ KHÔNG hợp lệ; các metric khác (MIA/Df/Dt/CE/time/GPU) vẫn đúng.")
    print(f"{'#'*72}\n")

    if not os.path.exists(forget_csv):
        raise FileNotFoundError(f"Forget set không tồn tại: {forget_csv}")

    # ---- Build override ----
    OVR = f"forget_set_path={forget_csv},id=loku_{forget_pct}per"
    if has_gold:
        OVR += f",retrained_model_path={retrained_path}"

    HYPOTHESIS = (f"exp11 honest (no F_re) tai FORGET {forget_pct}%: "
                  f"distill_teacher=og, early_stop=val, distill_forget=0, "
                  f"IHL={ihl}, image-FILA scale0.3. Multi-seed mean+-std.")

    # ---- Loop seeds ----
    for i, s in enumerate(seeds):
        if not force_redo and _seed_done(forget_pct, s):
            print(f"⏭️  SEED {s} đã có trong CSV — bỏ qua (set force_redo=True để chạy lại)\n")
            continue
        print(f"\n{'='*60}\n🎲 SEED {s}  ({i+1}/{len(seeds)})  ▸ FORGET {forget_pct}%\n{'='*60}")
        cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py '
               f'--config config.yaml --fresh --seed {s} '
               f'--override "{OVR}" '
               f'--exp {exp_name}_seed{s} --hypothesis "{HYPOTHESIS}"')
        get_ipython().system(cmd)

    # ---- Aggregate (truyền ihl xuống để in trong MD body) ----
    aggregate_summary(forget_pct, seeds, exp_name, paper, has_gold, ihl=ihl)


def aggregate_summary(forget_pct, seeds, exp_name, paper_ref, has_gold, ihl=0.75):
    """Đọc CSV → filter (forget%, seeds) → in bảng mean±std + so paper + lưu MD.

    Args:
        ihl (float): IHL weight để in vào MD body. Default 0.75 (sweet-spot).
                     Khi gọi standalone (không qua run_multiseed), pass đúng giá trị
                     đã dùng để train.
    """
    if not os.path.exists(CSV_PATH):
        print(f"❌ Không tìm thấy {CSV_PATH}. Bỏ qua aggregate.")
        return

    df = pd.read_csv(CSV_PATH)
    mask = df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per") & df['seed'].isin(seeds)
    df = df[mask]
    if df.empty:
        print(f"❌ Không có rows nào cho forget={forget_pct}% + seeds={list(seeds)}.")
        return

    # Lấy run mới nhất per seed (nếu có duplicate)
    if 'timestamp' in df.columns:
        df = df.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')

    # ---- Print table ----
    print(f"\n{'='*100}")
    print(f"📊 MULTI-SEED SUMMARY — FORGET {forget_pct}%, seeds={list(seeds)}")
    if not has_gold:
        print(f"⚠️  Gold retrained KHÔNG có cho {forget_pct}% → 1−CosSim KHÔNG hợp lệ")
    print(f"{'='*100}")
    hdr = (f"{'Metric':<20}" + "".join(f"{f'seed{s}':>10}" for s in seeds)
           + f"{'mean ± std':>18}" + f"{'paper':>10}" + f"{'Δ vs paper':>12}")
    print(hdr)
    print("-" * len(hdr))

    md_rows = [
        "| Metric | " + " | ".join(f"seed {s}" for s in seeds)
        + " | **mean ± std** | Paper | Δ vs paper |",
        "|---|" + "---|" * (len(seeds) + 3),
    ]

    summary = {}
    for csv_key, label, paper_key, direction in METRIC_DEFS:
        if csv_key not in df.columns:
            continue
        vals = []
        for s in seeds:
            sub = df[df['seed'] == s]
            if sub.empty:
                continue
            try:
                v = float(sub[csv_key].iloc[-1])
            except Exception:
                continue
            if v != v:  # NaN
                continue
            vals.append(v)
        if not vals:
            continue
        m, sd = float(np.mean(vals)), float(np.std(vals))
        summary[csv_key] = (m, sd)

        # Marker for invalid metrics
        invalid_marker = ""
        if csv_key == 'dist_vs_re' and not has_gold:
            invalid_marker = " ⚠️"

        # Compare vs paper
        if paper_key and paper_key in paper_ref:
            p_val = paper_ref[paper_key]
            delta = m - p_val
            arrow_good = (direction == '↓' and delta < 0) or (direction == '↑' and delta > 0)
            sym = "✅" if arrow_good else ("❌" if direction in ('↓','↑') else "·")
            paper_str = f"{p_val:>10.3f}"
            delta_str = f"{sym}{delta:+.3f}"
        else:
            paper_str = f"{'—':>10}"
            delta_str = "—"

        # Format seed values (pad if some missing)
        cell_vals = "".join(f"{v:>10.3f}" for v in vals)
        cell_vals += " " * (10 * (len(seeds) - len(vals)))

        # Print console row
        ms_str = f"{m:>10.3f}±{sd:.3f}"
        print(f"{label+invalid_marker:<20}{cell_vals}{ms_str:>18}{paper_str}{delta_str:>12}")

        # MD row
        md_vals = " | ".join(f"{v:.3f}" for v in vals) + " | " * (len(seeds) - len(vals))
        md_paper = f"{paper_ref[paper_key]:.3f}" if paper_key and paper_key in paper_ref else "—"
        md_delta = f"{delta:+.3f}" if paper_key and paper_key in paper_ref else "—"
        md_rows.append(f"| {label}{invalid_marker} | {md_vals} | **{m:.3f} ± {sd:.3f}** | {md_paper} | {md_delta} |")

    # ---- Save MD summary ----
    # New naming: summary_<slug>_multiseed.md (no "exp" prefix, no ID)
    os.makedirs("experiments", exist_ok=True)
    out_md = f"experiments/summary_{exp_name}_multiseed.md"
    paper_line = (f"**Paper Forget-MI ({forget_pct}%)**: "
                  f"MIA_paper={paper_ref['MIA_paper']} | "
                  f"Df_AUC={paper_ref['Df_AUC']} | Df_F1={paper_ref['Df_F1']} | "
                  f"Dt_AUC={paper_ref['Dt_AUC']} | Dt_F1={paper_ref['Dt_F1']} | "
                  f"Time≈{paper_ref['Time_h']}h")
    gold_line = ("**Gold retrained**: "
                 + (f"✅ available (`{GOLD_RETRAINED.get(forget_pct)}`)" if has_gold
                    else f"❌ N/A — 1−CosSim KHÔNG hợp lệ"))
    body = "\n".join([
        f"# Multi-seed FINAL — FORGET {forget_pct}% — {exp_name}",
        "",
        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_",
        "",
        f"**Config**: `final_ihl{int(ihl*100):03d}` (honest, no F_re), "
        f"distill_teacher=og, early_stop_metric=val, distill_forget_weight=0, IHL={ihl}",
        "",
        f"**Seeds**: {list(seeds)}",
        "",
        gold_line,
        "",
        *md_rows,
        "",
        paper_line,
        "",
        "_Δ vs paper_: âm = LoKU tốt hơn (đối với ↓ metrics) hoặc kém hơn (đối với ↑ metrics).",
        "Cụ thể: ✅ = LoKU thắng, ❌ = paper thắng.",
    ])
    with open(out_md, "w", encoding="utf-8") as f:
        f.write(body)
    print(f"\n💾 Summary MD: {out_md}")
    return summary


print("✅ Helpers đã định nghĩa: run_multiseed(), aggregate_summary()")
print("   Dùng: run_multiseed(3) / run_multiseed(6) / run_multiseed(10) ở các cell tiếp.")
print(f"   CSV path     : {CSV_PATH}")
print(f"   Gold retrained: 3% ✅ | 6% ❌ | 10% ❌")
print(f"   Naming       : exp file → exp_NNN_<slug>.md  |  summary → summary_<slug>_multiseed.md")

---

## 🎯 Training cells — chạy theo nhu cầu

Mỗi cell train multi-seed cho 1 forget percentage. Có thể chạy:
- **Một cell** (vd: chỉ 3%)
- **Tuần tự cả 3** để có data đầy đủ cho Bảng 1 luận văn
- **Lại một cell với `force_redo=True`** nếu muốn override seed cũ

Các cell **độc lập** — CSV chung (`unlearning_output/results_summary.csv`) không bị xóa giữa các lần chạy. Aggregator tự lọc theo `forget_pct` + `seed`.

In [ ]:
# ====================================
# CELL 4a: MULTI-SEED FORGET 3% — chính thức (gold ✅)
# ====================================
# 3% là dataset CHÍNH (có gold retrained → đầy đủ 12 metrics gồm 1−CosSim)
# Kết quả expected (theo experiments/summary_final_ihl075_multiseed.md):
#   MIA_paper ≈ 0.429 ± 0.116 (paper 0.571)
#   Df_AUC    ≈ 0.736 ± 0.004 (paper 0.735)
#   Dt_AUC    ≈ 0.677 ± 0.002 (paper 0.625)
#   Time      ≈ 0.20 ± 0.03 h (paper ~5h)
#
# Output:
#   • Individual: experiments/exp_NNN_final_ihl075_3per_seedXX.md (tự ID, không trùng prefix)
#   • Summary   : experiments/summary_final_ihl075_3per_multiseed.md

run_multiseed(forget_pct=3, seeds=(42, 123, 7), ihl=0.75)

In [ ]:
# ====================================
# CELL 4b: MULTI-SEED FORGET 6% — generalization check (⚠️ no gold)
# ====================================
# 6% chưa có model retrained → 1−CosSim KHÔNG hợp lệ.
# Các metric khác (MIA, Df/Dt_AUC, F1, CE, Time, GPU, params) vẫn đúng vì
# config exp11 honest KHÔNG dùng F_re khi train.
#
# Paper Forget-MI (6%): MIA=0.615 | Df_AUC=0.654 | Df_F1=0.328 | Dt_AUC=0.599 | Dt_F1=0.270

run_multiseed(forget_pct=6, seeds=(42, 123, 7), ihl=0.75)

In [ ]:
# ====================================
# CELL 4c: MULTI-SEED FORGET 10% — generalization check (⚠️ no gold)
# ====================================
# 10% chưa có model retrained → 1−CosSim KHÔNG hợp lệ.
# Paper báo: case khó nhất (forget nhiều nhất) → MIA cao, Test perf giảm.
#
# Paper Forget-MI (10%): MIA=0.810 | Df_AUC=0.656 | Df_F1=0.313 | Dt_AUC=0.565 | Dt_F1=0.252

run_multiseed(forget_pct=10, seeds=(42, 123, 7), ihl=0.75)

In [ ]:
# ====================================
# CELL 5: BẢNG 1 LUẬN VĂN — tổng hợp 3 forget% × paper vs LoKU
# ====================================
# Đọc CSV chung → lọc latest run per (forget%, seed) → in bảng so sánh 3 mức.
# Chạy sau khi đã hoàn thành cell 4a/4b/4c (có thể chạy partial — cell auto skip forget% chưa có).

import os, numpy as np, pandas as pd
from datetime import datetime

if not os.path.exists(CSV_PATH):
    print(f"❌ Không tìm thấy {CSV_PATH}. Cần chạy ít nhất 1 trong Cell 4a/4b/4c trước.")
else:
    df_all = pd.read_csv(CSV_PATH)
    pcts_in_csv = sorted({int(s.split('_')[-1].replace('per.csv','').replace('per',''))
                          for s in df_all['forget_pct'].dropna().astype(str).unique()
                          if '_' in s})

    print(f"📋 Tổng rows trong CSV: {len(df_all)}")
    print(f"📋 Forget% đã có data : {pcts_in_csv}")
    print()

    # ----- Metrics shown in summary table -----
    show_metrics = [
        ('MIA_paper',          'MIA',         '↓'),
        ('Df_AUC',             'Df_AUC',      '↓'),
        ('Df_F1',              'Df_F1',       '↓'),
        ('Dt_AUC',             'Dt_AUC',      '↑'),
        ('Dt_F1',              'Dt_F1',       '↑'),
        ('dist_vs_re',         '1−CosSim',    '↓'),
        ('unlearn_time_hours', 'Time(h)',     '↓'),
    ]

    md = [
        "# Bảng 1 luận văn — Forget-MI vs Forget-MI-LoKU (multi-forget%)",
        "",
        f"_Auto-generated từ `{CSV_PATH}` — {datetime.now().strftime('%Y-%m-%d %H:%M')}_",
        "",
        "⚠️ 1−CosSim cho 6% và 10% KHÔNG hợp lệ (chưa có gold retrained).",
        "",
    ]

    header = "| Forget% | Method | " + " | ".join(m[1] for m in show_metrics) + " |"
    sep = "|---|---|" + "---|" * len(show_metrics)
    md += [header, sep]

    print("=" * 110)
    print(header)
    print("=" * 110)

    for pct in [3, 6, 10]:
        sub = df_all[df_all['forget_pct'].astype(str).str.contains(f"_{pct}per")]
        if sub.empty:
            row = f"| {pct}% | _(chưa chạy)_ |" + " — |" * len(show_metrics)
            md.append(row)
            print(row)
            md.append("")
            continue

        if 'timestamp' in sub.columns:
            sub = sub.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')
        n_seeds = sub['seed'].nunique()
        has_gold_pct = _check_gold(pct)[1]

        # ----- Paper row -----
        paper_row = f"| {pct}% | Paper Forget-MI |"
        if pct in PAPER_REF:
            for csv_k, _, _ in show_metrics:
                # map csv_k → paper_key
                paper_key_map = {'MIA_paper': 'MIA_paper', 'Df_AUC': 'Df_AUC', 'Df_F1': 'Df_F1',
                                 'Dt_AUC': 'Dt_AUC', 'Dt_F1': 'Dt_F1', 'unlearn_time_hours': 'Time_h'}
                if csv_k in paper_key_map and paper_key_map[csv_k] in PAPER_REF[pct]:
                    paper_row += f" {PAPER_REF[pct][paper_key_map[csv_k]]:.3f} |"
                else:
                    paper_row += " — |"
        md.append(paper_row)
        print(paper_row)

        # ----- LoKU row -----
        loku_row = f"| {pct}% | **LoKU (n={n_seeds})** |"
        for csv_k, _, _ in show_metrics:
            if csv_k not in sub.columns:
                loku_row += " — |"
                continue
            vals = sub[csv_k].dropna().values
            if len(vals) == 0:
                loku_row += " — |"
                continue
            m, sd = float(np.mean(vals)), float(np.std(vals))
            # mark CosSim invalid for non-gold pct
            mark = "⚠️" if (csv_k == 'dist_vs_re' and not has_gold_pct) else ""
            loku_row += f" {m:.3f}±{sd:.3f}{mark} |"
        md.append(loku_row)
        print(loku_row)

        # ----- Δ row (LoKU - paper) -----
        delta_row = f"| {pct}% | Δ (LoKU − paper) |"
        if pct in PAPER_REF:
            for csv_k, _, direction in show_metrics:
                paper_key_map = {'MIA_paper': 'MIA_paper', 'Df_AUC': 'Df_AUC', 'Df_F1': 'Df_F1',
                                 'Dt_AUC': 'Dt_AUC', 'Dt_F1': 'Dt_F1', 'unlearn_time_hours': 'Time_h'}
                if csv_k not in sub.columns or csv_k not in paper_key_map or paper_key_map[csv_k] not in PAPER_REF[pct]:
                    delta_row += " — |"
                    continue
                vals = sub[csv_k].dropna().values
                if len(vals) == 0:
                    delta_row += " — |"
                    continue
                m = float(np.mean(vals))
                p = PAPER_REF[pct][paper_key_map[csv_k]]
                d = m - p
                good = (direction == '↓' and d < 0) or (direction == '↑' and d > 0)
                sym = "✅" if good else "❌"
                delta_row += f" {sym}{d:+.3f} |"
        md.append(delta_row)
        md.append("")
        print(delta_row)
        print("-" * 110)

    # ---- Save MD ----
    out_md = "experiments/bang1_loku_vs_paper_all_forgets.md"
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(md))
    print(f"\n💾 Bảng 1 saved: {out_md}")
    print("👉 Đây là BẢNG CHÍNH cho Chương 4 luận văn.")

In [ ]:
# ====================================
# CELL 6: Auto-commit & push experiment results lên GitHub
# ====================================
# Setup credentials (chọn 1 cách):
#   CÁCH A — Lưu vào Drive (KHUYẾN NGHỊ):
#     Tạo /content/drive/MyDrive/Forget-MI-Project/.git-secrets.json:
#       {"GITHUB_TOKEN":"ghp_...", "GIT_EMAIL":"...", "GIT_NAME":"..."}
#     Token: https://github.com/settings/tokens (scope: repo)
#   CÁCH B — Colab Secrets (sidebar 🔑): GITHUB_TOKEN, GIT_EMAIL, GIT_NAME
#   CÁCH C — Nhập tay mỗi session (fallback)

import os, json, getpass
from pathlib import Path

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

def load_secrets():
    drive_path = Path("/content/drive/MyDrive/Forget-MI-Project/.git-secrets.json")
    if drive_path.exists():
        s = json.loads(drive_path.read_text())
        print(f"🔑 Credentials từ {drive_path}")
        return s.get('GITHUB_TOKEN'), s.get('GIT_EMAIL'), s.get('GIT_NAME')
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            print("🔑 Credentials từ Colab Secrets")
            return t, userdata.get('GIT_EMAIL'), userdata.get('GIT_NAME')
    except Exception:
        pass
    if os.environ.get('GITHUB_TOKEN'):
        print("🔑 Credentials từ env vars")
        return (os.environ['GITHUB_TOKEN'],
                os.environ.get('GIT_EMAIL', ''),
                os.environ.get('GIT_NAME', ''))
    print("🔑 Nhập tay (sẽ ẩn token):")
    t = getpass.getpass("  GitHub token (ghp_...): ").strip()
    e = input("  Git email: ").strip()
    n = input("  Git name:  ").strip()
    return t, e, n

TOKEN, EMAIL, NAME = load_secrets()

if TOKEN and EMAIL and NAME:
    !git config user.email "{EMAIL}"
    !git config user.name "{NAME}"
    !git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git
    !git pull --rebase origin {BRANCH} 2>&1 | tail -5
    !git add experiments/ 2>/dev/null
    changes = !git diff --cached --name-only
    if changes and any(c.strip() for c in changes):
        print("\n📦 Files sẽ commit:")
        for f in changes:
            if f.strip():
                print(f"   - {f}")
        commit_msg = "exp results: auto-tracked multi-seed (3/6/10%)"
        !git commit -m "{commit_msg}"
        !git push origin {BRANCH}
        print(f"\n✅ Pushed: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
    else:
        print("ℹ️  Không có file experiment mới.")
else:
    print("⚠️  Thiếu credentials — bỏ qua push.")